In [152]:
import requests
import msal
import os
from openai import OpenAI
import json
from dotenv import load_dotenv

In [153]:
load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID")
CHATGPT_API_KEY = os.getenv("CHATGPT_API_KEY")

In [ ]:
SCOPES = ["https://graph.microsoft.com/Mail.Read"]
CACHE_FILE = "token_cache.bin"

In [140]:
def get_token():
    cache = msal.SerializableTokenCache()
    if os.path.exists(CACHE_FILE):
        cache.deserialize(open(CACHE_FILE, "r").read())
        print("--- Found existing cache file ---")

    app = msal.PublicClientApplication(
        CLIENT_ID, 
        authority="https://login.microsoftonline.com/common",
        token_cache=cache
    )

    accounts = app.get_accounts()
    result = None
    
    if accounts:
        print(f"--- Attempting silent login for: {accounts[0]['username']} ---")
        result = app.acquire_token_silent(SCOPES, account=accounts[0])

    if not result:
        print("--- Silent login failed. Triggering Device Flow... ---")
        flow = app.initiate_device_flow(scopes=SCOPES)
        print(flow["message"])
        result = app.acquire_token_by_device_flow(flow)

        if "access_token" in result:
            with open(CACHE_FILE, "w") as f:
                f.write(cache.serialize())
            print("--- Success! Cache updated for future use ---")

    return result.get("access_token")

In [150]:
def fetch_emails_as_json():
    emails = []

    token = get_token()
    if not token:
        return print("Failed to get token.")

    print("\n[Connected] Fetching recent emails...")
    headers = {"Authorization": f"Bearer {token}"}

    res = requests.get("https://graph.microsoft.com/v1.0/me/messages?$top=3", headers=headers).json()
    
    for msg in res.get("value", []):
        emails.append(msg)
    
    return emails

In [148]:
def format_email_for_ai(email_json):
    """
    Strips a Microsoft Graph email JSON down to the essential 
    bits to save tokens and improve AI focus.
    """
    sender_name = email_json.get('from', {}).get('emailAddress', {}).get('name', 'Unknown')
    sender_email = email_json.get('from', {}).get('emailAddress', {}).get('address', '')
    subject = email_json.get('subject', 'No Subject')
    date_received = email_json.get('receivedDateTime', '')
    email_id = email_json.get('id', '')

    to_me = any(
        rec.get('emailAddress', {}).get('address') == 'jonesynathan@outlook.com'
        for rec in email_json.get('toRecipients', [])
    )
    role = "DIRECT" if to_me else "CC/BCC"

    content = email_json.get('bodyPreview', '')
    clean_content = content.encode('ascii', 'ignore').decode('ascii')
    clean_content = " ".join(clean_content.split())

    formatted_block = (
        f"EMAIL_ID: {email_id}\n"
        f"FROM: {sender_name} ({sender_email})\n"
        f"DATE: {date_received}\n"
        f"LEVEL: {role}\n"
        f"SUBJECT: {subject}\n"
        f"CONTENT: {clean_content}\n"
    )

    return formatted_block

In [143]:
def fetch_emails():
    """
    Returns a list of strings describing each email.
    Email strings come from format_email_for_ai
    """
    emails = []

    token = get_token()
    if not token:
        return print("Failed to get token.")

    print("\n[Connected] Fetching recent emails...")
    headers = {"Authorization": f"Bearer {token}"}

    res = requests.get("https://graph.microsoft.com/v1.0/me/messages?$top=10", headers=headers).json()
    
    for msg in res.get("value", []):
        emails.append(format_email_for_ai(msg))
    
    return emails

In [144]:
client = OpenAI(api_key=CHATGPT_API_KEY)

In [145]:
import json
from datetime import datetime

def process_emails(formatted_emails):
    current_time = datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

    system_prompt = """
You are an AI executive assistant that processes one email at a time.

Return ONLY a valid JSON object.

STRICT RULES:
- Do NOT include markdown
- Do NOT include ```json
- Do NOT include any explanation
- Output must start with { and end with }
- All fields must be present
- If no response is needed, set draft_reply to null

Schema:
{
  "email_id": "...",
  "needs_response": true,
  "priority": "high | medium | low",
  "category": "action | informational | marketing | spam",
  "summary": "...",
  "action_items": [],
  "draft_reply": null,
  "suggested_followup_time": "none | today | this_week | later",
  "confidence": 0.0
}

Additional rules:
- If the email is marketing or spam, set needs_response to false
- Only generate draft_reply if needs_response is true
- Keep summaries concise (1–2 sentences)
- Extract clear action items when present
- Confidence must be between 0 and 1
"""

    email_results = []

    for email in formatted_emails:

        user_content = f"""
Current Time: {current_time}

Email:
{email}
"""

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ],
            temperature=0.2,
            max_tokens=800
        )

        content = response.choices[0].message.content.strip()

        result = safe_parse_json(content, retries=1)
        if result:
            email_results.append(result)
        else:
            print("⚠️ Failed after retry:", content)

    return email_results

In [ ]:
def safe_parse_json(content, retries=1):
    import json

    for _ in range(retries + 1):
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            pass
    return None

In [149]:
formatted_emails = fetch_emails()
result = process_emails(formatted_emails)

--- Found existing cache file ---
--- Attempting silent login for: jonesynathan@outlook.com ---

[Connected] Fetching recent emails...


In [ ]:
for email in email_results:
    if email["needs_response"] and email["draft_reply"]:
        create_draft(email["email_id"], email["draft_reply"])